In [24]:
import os
import streamlit as st
import time
import langchain
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import UnstructuredURLLoader

In [34]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.reuters.com/markets/adani-vs-hindenburg-what-you-need-know-2023-01-31/"
])
data = loaders.load()
len(data)

1

In [42]:
data = loaders.load()

In [36]:
from dotenv import load_dotenv
load_dotenv()
chatllm = ChatOpenAI(temperature=0.2,model_name = "gpt-3.5-turbo", max_tokens = 500)

In [37]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

docs = splitter.split_documents(data)
len(docs)

1

In [ ]:
print(docs)

In [38]:
import pickle
embeddings = OpenAIEmbeddings()
vectorindex_openai = FAISS.from_documents(docs,embeddings)
file_path = "vector_index.pkl"
with open(file_path, "wb") as f:
    pickle.dump(vectorindex_openai, f)
    

In [39]:
import pickle
if os.path.exists(file_path):
    with open(file_path, 'rb') as f:
        vectorIndex = pickle.load(f)

In [ ]:
chain = RetrievalQAWithSourcesChain.from_llm(llm = chatllm , retriever = vectorIndex.as_retriever())
chain

In [ ]:

query = "what happened between adani and heidenburg"

langchain.debug = True
chain({"question": query},return_only_outputs = True)

In [ ]:
query = "what is the price of Tiago iCNG?"

langchain.debug = True
chain({"question":query},return_only_outputs = True)